# Exploración del latente y del DMD

Plantilla del change `ae-reconstruccion` (tarea 8.5), con dos secciones:

- **Nivel 1 (sin TensorFlow):** `latent.parquet` y `dmd_operator.npz`: dimensiones
  activas, trayectorias de `z_mp` y `z_aux`, reajuste del DMD con otro rango o
  ventana (numpy), espectro de `K` (período y decaimiento) y linealizabilidad por época.
- **Nivel 2 (con TensorFlow):** `encoder.keras` y `decoder_mp.keras`: pronóstico del
  DMD a µg/m³, recorrer una dimensión del latente, y anular `z_aux` o un modo.

Insumo exploratorio para `ae-dmd`; no es un resultado final.

Gráficos con matplotlib (no Plotly: en el Space las figuras interactivas tardan demasiado).

In [ ]:
import sys, pathlib
# Raíz de importación del pipeline: el notebook vive en notebooks/ (o en la raíz del repo).
_here = pathlib.Path.cwd()
PIPELINE = next(p for p in (_here / "pipeline", _here.parent / "pipeline") if p.exists())
sys.path.insert(0, str(PIPELINE))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analysis import load
from features.build import load_validated
from ae import data as aed
from ae import dmd as dmdm
from ae import metrics as aem

BUCKET_DATA = "sinca-data"
# Credenciales: el rol del Space (sinca-dev-role) las provee; no escribir claves aquí.

## 1. Elegir run AE y leer el latente y el operador (nivel 1)

In [ ]:
state = load.ensure_state()
runs = load.list_runs(state)
ae_runs = load.ae_runs(runs)
RUN_NAME = ae_runs[-1]["run_name"]
run = next(r for r in ae_runs if r["run_name"] == RUN_NAME)
arts = load.load_artifacts(state, run["run_id"])
latent = arts["latent.parquet"]
op_path = load.artifact_path(state, run["run_id"], "dmd_operator.npz")
op = np.load(op_path) if op_path else None
print("latente:", latent.shape, "| operador:", "sí" if op is not None else "no (sin pase DMD)")

### Dimensiones activas y trayectorias

In [ ]:
zcols = [c for c in latent.columns if c.startswith("z_")]
display(latent[zcols].std().round(4).to_frame("std"))
fig, ax = plt.subplots(figsize=(12, 4))
for c in zcols[:6]:
    ax.plot(latent["timestamp"], latent[c], lw=0.6, label=c)
ax.set_title("Trayectorias del latente (primeras 6 dimensiones)")
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()

### Espectro de K: autovalores → período y decaimiento (por hora)

In [ ]:
if op is not None:
    eig = op["eigvals_real"] + 1j * op["eigvals_imag"]
    decay = -np.log(np.abs(eig) + 1e-12)            # > 0 decae, < 0 crece
    period = np.where(np.abs(np.angle(eig)) > 1e-9, 2 * np.pi / np.abs(np.angle(eig)), np.inf)
    display(pd.DataFrame({"|lambda|": np.abs(eig), "decaimiento_h": decay, "periodo_h": period}).round(4))

### Reajustar el DMD con otro rango o ventana (numpy, sin TF)

Parte el latente de train en segmentos de horas consecutivas (`dmdm.split_segments`)
y reajusta `K` con el rango que se quiera. Se compara la linealizabilidad en test.

In [ ]:
RANK = 4          # probar otros rangos
train = latent[latent["split"] == "train"] if (latent["split"] == "train").any() else latent
test = latent[latent["split"] == "test"]
positions = np.arange(len(train))
segs_tr = dmdm.split_segments(train[zcols].to_numpy(), positions)
fit_r = dmdm.fit_dmd(segs_tr, rank=RANK)
lin = dmdm.linearizability(fit_r["K"], [test[zcols].to_numpy()])
print("rango", fit_r["rank"], "| linealizabilidad test:", {k: round(v, 4) for k, v in lin.items()})

### Linealizabilidad por época del año (error de 1 paso)

In [ ]:
K = fit_r["K"]
seg = latent[zcols].to_numpy()
pred = seg[:-1] @ K.T
err = np.sqrt(((pred - seg[1:]) ** 2).mean(axis=1))
by_month = pd.DataFrame({"mes": latent["timestamp"].dt.month.to_numpy()[1:], "rmse_1paso": err}).groupby("mes")["rmse_1paso"].mean()
by_month.plot.bar(figsize=(8, 3.5), title="RMSE de 1 paso por mes (K ajustado en train)")
plt.tight_layout()
plt.show()

## Nivel 2 — requiere TensorFlow

Instalar `requirements-notebooks-tf.txt` en el venv del Space **solo si se va a usar esta
sección** (verificar primero si TF ya viene en la imagen; tarea 9.4). Los `.keras` se
guardan por separado y se cargan sin compilar.

In [ ]:
# Nivel 2: requiere TensorFlow (ver requirements-notebooks-tf.txt).
try:
    import keras
    HAS_TF = True
except ImportError:
    HAS_TF = False
    print("TensorFlow no instalado: instalar requirements-notebooks-tf.txt (solo nivel 2).")

if HAS_TF:
    enc_path = load.artifact_path(state, run["run_id"], "encoder.keras")
    dec_path = load.artifact_path(state, run["run_id"], "decoder_mp.keras")
    encoder = keras.saving.load_model(enc_path, compile=False) if enc_path else None
    decoder = keras.saving.load_model(dec_path, compile=False) if dec_path else None
    print("encoder:", encoder is not None, "| decoder_mp:", decoder is not None)

### Recorrer una dimensión del latente y ver su efecto sobre las 6 series

In [ ]:
if HAS_TF and decoder is not None:
    d = latent[zcols].shape[1]
    z_mean = latent[zcols].mean().to_numpy()
    dim = 0
    steps = np.linspace(-2, 2, 5) * latent[zcols[dim]].std()
    zs = np.repeat(z_mean[None, :], len(steps), axis=0)
    zs[:, dim] = z_mean[dim] + steps
    decoded = decoder.predict(zs, verbose=0)          # [pasos, L, 6] en z-score
    # Para µg/m³ hace falta el normalizador de train (ver ae.run.make_context).
    print("ventanas decodificadas:", decoded.shape)

### Anular z_aux y ver el efecto sobre el decoder de MP

In [ ]:
if HAS_TF and decoder is not None:
    d_aux = int(sum(c.startswith("z_aux_") for c in zcols))
    base = decoder.predict(latent[zcols].to_numpy()[:200], verbose=0)
    z_ab = latent[zcols].to_numpy()[:200].copy()
    z_ab[:, -d_aux:] = 0.0
    ablated = decoder.predict(z_ab, verbose=0)
    print("diferencia media absoluta (z-score) al anular z_aux:", float(np.abs(base - ablated).mean()))